<a href="https://colab.research.google.com/github/nexageapps/AI/blob/main/Basic/B10a%20-%20Recurrent%20Neural%20Networks%20(COMPSCI%20714).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# B10a: Recurrent Neural Networks - COMPSCI 714 Lecture 4

**Course:** COMPSCI 714 - AI Architecture and Design  
**Lecture:** 4 - Learning with Sequences - RNNs

---

## Important Disclaimer

This notebook is an **independent educational resource** created to help students understand RNN concepts. This is **NOT official University of Auckland course material** and is not affiliated with or endorsed by the university.

**Use Responsibly:**
- This is a learning companion, not a substitute for lectures
- Understand concepts first, then implement independently for assignments
- Follow the University of Auckland's Academic Integrity Policy
- Use for learning and understanding, not for copying solutions

---

## Learning Outcomes

This notebook covers COMPSCI 714 Lecture 4 topics:

1. **Sequential Data** - what makes data sequential and why it matters
2. **Various Sequence Tasks** - one-to-one, one-to-many, many-to-one, many-to-many
3. **Dealing with Sequences** - challenges and approaches
4. **Typical RNN Architecture** - hidden state, weight sharing, unrolling
5. **RNN Process** - forward pass step by step
6. **Several Types of RNNs** - Vanilla, LSTM, GRU
7. **Character-Level Language Model** - worked example
8. **Backpropagation Through Time (BPTT)** - training RNNs
9. **Deep RNNs** - stacking several layers
10. **Visualising RNNs** - understanding what RNNs learn

## Prerequisites

- B05a - Neural Networks Theory (COMPSCI 714 Lecture 2)
- B05b - Training and Optimization (COMPSCI 714 Lecture 3)
- B10 - Recurrent Neural Networks (general)
- Understanding of backpropagation and gradient descent

### Interactive App
- [RNN Explorer](https://nexageapps.github.io/AI/rnn-explorer/) - Visualise RNN data flow, gates, BPTT, and vanishing gradients

### Extra Resources
- [Understanding LSTM Networks (Colah's Blog)](http://colah.github.io/posts/2015-08-Understanding-LSTMs/)
- [The Unreasonable Effectiveness of RNNs (Karpathy)](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)
- [Visualizing and Understanding Recurrent Networks (Karpathy et al.)](https://arxiv.org/abs/1506.02078)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
print('Notebook ready!')

---

# Part 1: Sequential Data

## 1.1 What is Sequential Data?

Sequential data is data where **order matters**. The meaning or value of each element depends on what came before (and sometimes after) it.

**Key property:** Temporal or positional dependencies between elements.

### Examples of Sequential Data

| Domain | Example | Why Order Matters |
|--------|---------|-------------------|
| **Text** | "The cat sat on the mat" | Word order determines meaning |
| **Speech** | Audio waveform | Temporal structure encodes phonemes |
| **Time Series** | Stock prices, weather | Past values predict future |
| **Video** | Sequence of frames | Motion requires frame ordering |
| **DNA** | ATCGGATC... | Gene sequences encode proteins |
| **Music** | Note sequences | Melody depends on note order |

### Why Standard Neural Networks Fail on Sequences

1. **Fixed input size** - sequences have variable length
2. **No parameter sharing** - features learned at position 1 aren't reused at position 10
3. **No memory** - each input processed independently, no notion of "what came before"

> **Key Insight:** We need an architecture that can process inputs one at a time while maintaining a *memory* of what it has seen so far.

In [ ]:
# Demonstrate why order matters in sequential data
sentence_1 = "dog bites man"
sentence_2 = "man bites dog"

print("Same words, different order, different meaning:")
print(f"  Sentence 1: '{sentence_1}'")
print(f"  Sentence 2: '{sentence_2}'")
print()

# Time series example: order encodes the trend
prices = [100, 102, 105, 103, 108, 112]
prices_shuffled = [105, 100, 112, 103, 108, 102]

print("Stock prices in order show an upward trend:")
print(f"  Ordered:  {prices}")
print(f"  Shuffled: {prices_shuffled} (trend is lost)")

## 1.2 Various Sequence Tasks

RNNs can handle different input-output configurations:

### Task Types

```
1. One-to-One       2. One-to-Many       3. Many-to-One       4. Many-to-Many      5. Many-to-Many
   (standard)        (generation)          (classification)     (synced)              (encoder-decoder)

   x → [H] → y      x → [H] → y₁        x₁ → [H]            x₁ → [H] → y₁       x₁ → [H]
                          [H] → y₂        x₂ → [H]            x₂ → [H] → y₂            [H]
                          [H] → y₃        x₃ → [H] → y        x₃ → [H] → y₃            [H] → y₁
                                                                                          [H] → y₂
```

| Type | Input | Output | Example |
|------|-------|--------|---------|
| **One-to-One** | Single | Single | Image classification (not really RNN) |
| **One-to-Many** | Single | Sequence | Image captioning, music generation |
| **Many-to-One** | Sequence | Single | Sentiment analysis, video classification |
| **Many-to-Many (synced)** | Sequence | Sequence (same length) | POS tagging, named entity recognition |
| **Many-to-Many (encoder-decoder)** | Sequence | Sequence (different length) | Machine translation, summarisation |

In [ ]:
# Visualise the different RNN task types
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

task_names = ['One-to-Many', 'Many-to-One', 'Many-to-Many\n(synced)', 'Many-to-Many\n(enc-dec)']
task_examples = ['Image captioning', 'Sentiment analysis', 'POS tagging', 'Translation']

# Colour scheme
c_input = '#4CAF50'
c_hidden = '#2196F3'
c_output = '#FF9800'

for idx, (ax, name, example) in enumerate(zip(axes, task_names, task_examples)):
    ax.set_xlim(-0.5, 4.5)
    ax.set_ylim(-0.5, 3.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f'{name}\n({example})', fontsize=10, fontweight='bold')

    if idx == 0:  # One-to-Many
        ax.add_patch(plt.Rectangle((0.5, 0), 0.8, 0.8, fc=c_input, ec='black'))
        ax.text(0.9, 0.4, 'x', ha='center', va='center', fontsize=10, color='white')
        for i in range(3):
            ax.add_patch(plt.Rectangle((0.5+i*1.2, 1.2), 0.8, 0.8, fc=c_hidden, ec='black'))
            ax.text(0.9+i*1.2, 1.6, f'h{i+1}', ha='center', va='center', fontsize=9, color='white')
            ax.add_patch(plt.Rectangle((0.5+i*1.2, 2.4), 0.8, 0.8, fc=c_output, ec='black'))
            ax.text(0.9+i*1.2, 2.8, f'y{i+1}', ha='center', va='center', fontsize=9, color='white')
            ax.annotate('', xy=(0.9+i*1.2, 2.4), xytext=(0.9+i*1.2, 2.0), arrowprops=dict(arrowstyle='->', color='black'))
            if i > 0:
                ax.annotate('', xy=(0.5+i*1.2, 1.6), xytext=(0.5+(i-1)*1.2+0.8, 1.6), arrowprops=dict(arrowstyle='->', color='black'))
        ax.annotate('', xy=(0.9, 1.2), xytext=(0.9, 0.8), arrowprops=dict(arrowstyle='->', color='black'))

    elif idx == 1:  # Many-to-One
        for i in range(3):
            ax.add_patch(plt.Rectangle((0.5+i*1.2, 0), 0.8, 0.8, fc=c_input, ec='black'))
            ax.text(0.9+i*1.2, 0.4, f'x{i+1}', ha='center', va='center', fontsize=9, color='white')
            ax.add_patch(plt.Rectangle((0.5+i*1.2, 1.2), 0.8, 0.8, fc=c_hidden, ec='black'))
            ax.text(0.9+i*1.2, 1.6, f'h{i+1}', ha='center', va='center', fontsize=9, color='white')
            ax.annotate('', xy=(0.9+i*1.2, 1.2), xytext=(0.9+i*1.2, 0.8), arrowprops=dict(arrowstyle='->', color='black'))
            if i > 0:
                ax.annotate('', xy=(0.5+i*1.2, 1.6), xytext=(0.5+(i-1)*1.2+0.8, 1.6), arrowprops=dict(arrowstyle='->', color='black'))
        ax.add_patch(plt.Rectangle((0.5+2*1.2, 2.4), 0.8, 0.8, fc=c_output, ec='black'))
        ax.text(0.9+2*1.2, 2.8, 'y', ha='center', va='center', fontsize=10, color='white')
        ax.annotate('', xy=(0.9+2*1.2, 2.4), xytext=(0.9+2*1.2, 2.0), arrowprops=dict(arrowstyle='->', color='black'))

    elif idx == 2:  # Many-to-Many synced
        for i in range(3):
            ax.add_patch(plt.Rectangle((0.5+i*1.2, 0), 0.8, 0.8, fc=c_input, ec='black'))
            ax.text(0.9+i*1.2, 0.4, f'x{i+1}', ha='center', va='center', fontsize=9, color='white')
            ax.add_patch(plt.Rectangle((0.5+i*1.2, 1.2), 0.8, 0.8, fc=c_hidden, ec='black'))
            ax.text(0.9+i*1.2, 1.6, f'h{i+1}', ha='center', va='center', fontsize=9, color='white')
            ax.add_patch(plt.Rectangle((0.5+i*1.2, 2.4), 0.8, 0.8, fc=c_output, ec='black'))
            ax.text(0.9+i*1.2, 2.8, f'y{i+1}', ha='center', va='center', fontsize=9, color='white')
            ax.annotate('', xy=(0.9+i*1.2, 1.2), xytext=(0.9+i*1.2, 0.8), arrowprops=dict(arrowstyle='->', color='black'))
            ax.annotate('', xy=(0.9+i*1.2, 2.4), xytext=(0.9+i*1.2, 2.0), arrowprops=dict(arrowstyle='->', color='black'))
            if i > 0:
                ax.annotate('', xy=(0.5+i*1.2, 1.6), xytext=(0.5+(i-1)*1.2+0.8, 1.6), arrowprops=dict(arrowstyle='->', color='black'))

    elif idx == 3:  # Many-to-Many encoder-decoder
        for i in range(2):
            ax.add_patch(plt.Rectangle((0.2+i*1.0, 0), 0.7, 0.7, fc=c_input, ec='black'))
            ax.text(0.55+i*1.0, 0.35, f'x{i+1}', ha='center', va='center', fontsize=8, color='white')
            ax.add_patch(plt.Rectangle((0.2+i*1.0, 1.0), 0.7, 0.7, fc=c_hidden, ec='black'))
            ax.text(0.55+i*1.0, 1.35, f'h{i+1}', ha='center', va='center', fontsize=8, color='white')
            ax.annotate('', xy=(0.55+i*1.0, 1.0), xytext=(0.55+i*1.0, 0.7), arrowprops=dict(arrowstyle='->', color='black', lw=0.8))
            if i > 0:
                ax.annotate('', xy=(0.2+i*1.0, 1.35), xytext=(0.2+(i-1)*1.0+0.7, 1.35), arrowprops=dict(arrowstyle='->', color='black', lw=0.8))
        for i in range(2):
            ax.add_patch(plt.Rectangle((2.5+i*1.0, 1.0), 0.7, 0.7, fc=c_hidden, ec='black'))
            ax.text(2.85+i*1.0, 1.35, f'd{i+1}', ha='center', va='center', fontsize=8, color='white')
            ax.add_patch(plt.Rectangle((2.5+i*1.0, 2.2), 0.7, 0.7, fc=c_output, ec='black'))
            ax.text(2.85+i*1.0, 2.55, f'y{i+1}', ha='center', va='center', fontsize=8, color='white')
            ax.annotate('', xy=(2.85+i*1.0, 2.2), xytext=(2.85+i*1.0, 1.7), arrowprops=dict(arrowstyle='->', color='black', lw=0.8))
            if i > 0:
                ax.annotate('', xy=(2.5+i*1.0, 1.35), xytext=(2.5+(i-1)*1.0+0.7, 1.35), arrowprops=dict(arrowstyle='->', color='black', lw=0.8))
        ax.annotate('', xy=(2.5, 1.35), xytext=(1.9, 1.35), arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

plt.suptitle('RNN Task Types', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 1.3 Dealing with Sequences - Challenges

### Challenge 1: Variable Length
Sequences can be different lengths. Solutions:
- **Padding:** Add zeros to make all sequences the same length
- **Masking:** Tell the model to ignore padded positions
- **Bucketing:** Group similar-length sequences together

### Challenge 2: Long-Range Dependencies
Information from early in a sequence may be needed much later.
- "The cat, which sat on the mat and ate the fish, **was** happy" (was agrees with cat, not fish)

### Challenge 3: Computational Cost
Processing long sequences step-by-step is inherently sequential (hard to parallelise).

### Why Not Just Use a Feedforward Network?
- Would need to flatten the sequence → loses ordering information
- Fixed input size → can't handle variable-length sequences
- No weight sharing across time → inefficient, doesn't generalise across positions

---

# Part 2: Typical RNN Architecture

## 2.1 The Core Idea

An RNN processes a sequence **one element at a time**, maintaining a **hidden state** `h` that acts as memory.

At each time step `t`:
1. Take input `x_t`
2. Combine with previous hidden state `h_{t-1}`
3. Produce new hidden state `h_t`
4. Optionally produce output `y_t`

### The RNN Equations

```
h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b_h)
y_t = W_hy · h_t + b_y
```

Where:
- `W_xh`: Input-to-hidden weights (same at every time step!)
- `W_hh`: Hidden-to-hidden weights (same at every time step!)
- `W_hy`: Hidden-to-output weights
- `b_h`, `b_y`: Biases
- `tanh`: Activation function (squashes to [-1, 1])

> **Key Insight:** The **same weights** are used at every time step. This is called **weight sharing** or **parameter sharing** - it's what allows RNNs to generalise across different positions in a sequence.

### Unrolling Through Time

The recurrent connection can be "unrolled" to visualise the computation:

```
h_0    h_1    h_2    h_3
 ↑      ↑      ↑      ↑
[W]    [W]    [W]    [W]    ← Same weights W at each step
 ↑      ↑      ↑      ↑
x_0    x_1    x_2    x_3

h_0 → [W_hh] → h_1 → [W_hh] → h_2 → [W_hh] → h_3
```

In [ ]:
# Implement a vanilla RNN cell from scratch
class VanillaRNNCell:
    """A single RNN cell: h_t = tanh(W_hh * h_{t-1} + W_xh * x_t + b)"""

    def __init__(self, input_size, hidden_size):
        # Initialise weights (Xavier initialisation)
        scale_xh = np.sqrt(2.0 / (input_size + hidden_size))
        scale_hh = np.sqrt(2.0 / (hidden_size + hidden_size))

        self.W_xh = np.random.randn(input_size, hidden_size) * scale_xh
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale_hh
        self.b_h = np.zeros(hidden_size)
        self.hidden_size = hidden_size

    def forward(self, x_t, h_prev):
        """One step of the RNN."""
        # Pre-activation: combine input and previous hidden state
        z = x_t @ self.W_xh + h_prev @ self.W_hh + self.b_h
        # Activation
        h_t = np.tanh(z)
        return h_t

    def forward_sequence(self, X):
        """Process an entire sequence, return all hidden states."""
        T = X.shape[0]  # sequence length
        h = np.zeros(self.hidden_size)  # initial hidden state
        hidden_states = [h.copy()]

        for t in range(T):
            h = self.forward(X[t], h)
            hidden_states.append(h.copy())

        return np.array(hidden_states)

# Demo: process a short sequence
input_size = 4
hidden_size = 3
seq_length = 5

rnn = VanillaRNNCell(input_size, hidden_size)

# Random input sequence (5 time steps, 4 features each)
X = np.random.randn(seq_length, input_size)

print("RNN Forward Pass (step by step):")
print(f"  Input shape: ({seq_length}, {input_size})")
print(f"  Hidden size: {hidden_size}")
print(f"  W_xh shape: {rnn.W_xh.shape}")
print(f"  W_hh shape: {rnn.W_hh.shape}")
print()

hidden_states = rnn.forward_sequence(X)

for t in range(seq_length + 1):
    label = "h_0 (initial)" if t == 0 else f"h_{t} (after x_{t})"
    print(f"  {label}: {np.round(hidden_states[t], 3)}")

## 2.2 RNN Process - Step by Step

Let's trace through exactly what happens at each time step:

**Step 1:** Initialise hidden state `h_0 = 0` (vector of zeros)

**Step 2:** At time `t = 1`:
- Read input `x_1`
- Compute `z_1 = W_xh · x_1 + W_hh · h_0 + b_h`
- Apply activation: `h_1 = tanh(z_1)`
- (Optional) Compute output: `y_1 = W_hy · h_1 + b_y`

**Step 3:** At time `t = 2`:
- Read input `x_2`
- Compute `z_2 = W_xh · x_2 + W_hh · h_1 + b_h`  ← uses h_1 from previous step!
- Apply activation: `h_2 = tanh(z_2)`

**...and so on for each time step.**

> **The hidden state `h_t` encodes a summary of everything the network has seen up to time `t`.**

In [ ]:
# Visualise how hidden state evolves over a sequence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: hidden state values over time
ax = axes[0]
for dim in range(hidden_size):
    ax.plot(range(seq_length + 1), hidden_states[:, dim],
            marker='o', label=f'h[{dim}]', linewidth=2)
ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Hidden State Value', fontsize=12)
ax.set_title('Hidden State Evolution Over Time', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(range(seq_length + 1))
ax.set_xticklabels(['h₀'] + [f't={t}' for t in range(1, seq_length + 1)])

# Right: heatmap of hidden states
ax = axes[1]
im = ax.imshow(hidden_states.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Hidden Dimension', fontsize=12)
ax.set_title('Hidden State Heatmap', fontsize=13, fontweight='bold')
ax.set_xticks(range(seq_length + 1))
ax.set_xticklabels(['h₀'] + [f't={t}' for t in range(1, seq_length + 1)])
ax.set_yticks(range(hidden_size))
ax.set_yticklabels([f'h[{i}]' for i in range(hidden_size)])
plt.colorbar(im, ax=ax, label='Activation value')

plt.tight_layout()
plt.show()

print("Notice: hidden state values are bounded in [-1, 1] due to tanh activation")
print("Each dimension captures different aspects of the input sequence")

---

# Part 3: Several Types of RNNs

## 3.1 Vanilla RNN

The basic RNN we just implemented. Simple but has a critical flaw: **vanishing gradients**.

```
h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b_h)
```

**Problem:** When training on long sequences, gradients get multiplied by `W_hh` at each time step during backpropagation. If the largest eigenvalue of `W_hh` is < 1, gradients shrink exponentially → **vanishing gradient**. If > 1, they explode → **exploding gradient**.

## 3.2 LSTM (Long Short-Term Memory)

Hochreiter & Schmidhuber (1997) introduced LSTM to solve the vanishing gradient problem.

**Key idea:** Add a **cell state** `C_t` that acts as a "conveyor belt" - information can flow along it unchanged. **Gates** control what information to add or remove.

### LSTM Gates

| Gate | Formula | Purpose |
|------|---------|---------|
| **Forget** | `f_t = σ(W_f · [h_{t-1}, x_t] + b_f)` | What to **remove** from cell state |
| **Input** | `i_t = σ(W_i · [h_{t-1}, x_t] + b_i)` | What new info to **store** |
| **Candidate** | `C̃_t = tanh(W_C · [h_{t-1}, x_t] + b_C)` | New candidate values |
| **Cell update** | `C_t = f_t ⊙ C_{t-1} + i_t ⊙ C̃_t` | Updated cell state |
| **Output** | `o_t = σ(W_o · [h_{t-1}, x_t] + b_o)` | What to **output** |
| **Hidden** | `h_t = o_t ⊙ tanh(C_t)` | New hidden state |

> **Why it works:** The cell state update `C_t = f_t ⊙ C_{t-1} + i_t ⊙ C̃_t` is an **additive** operation. Gradients flow through addition without shrinking, solving the vanishing gradient problem.

## 3.3 GRU (Gated Recurrent Unit)

Cho et al. (2014) simplified LSTM into GRU with only **2 gates** (vs LSTM's 3):

| Gate | Formula | Purpose |
|------|---------|---------|
| **Reset** | `r_t = σ(W_r · [h_{t-1}, x_t] + b_r)` | How much past to forget |
| **Update** | `z_t = σ(W_z · [h_{t-1}, x_t] + b_z)` | Balance old vs new |
| **Candidate** | `h̃_t = tanh(W · [r_t ⊙ h_{t-1}, x_t] + b)` | New candidate |
| **Hidden** | `h_t = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ h̃_t` | Interpolate old and new |

### Comparison

| Feature | Vanilla RNN | LSTM | GRU |
|---------|-------------|------|-----|
| **Gates** | 0 | 3 (forget, input, output) | 2 (reset, update) |
| **States** | h only | h + cell state C | h only |
| **Parameters** | Fewest | Most (~4x vanilla) | Middle (~3x vanilla) |
| **Long-range deps** | Poor | Excellent | Good |
| **Training speed** | Fastest | Slowest | Middle |

In [ ]:
# Implement LSTM cell from scratch to understand the gates
class LSTMCell:
    """LSTM cell with forget, input, and output gates."""

    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        combined = input_size + hidden_size
        scale = np.sqrt(2.0 / combined)

        # All four weight matrices (forget, input, candidate, output)
        self.W_f = np.random.randn(combined, hidden_size) * scale
        self.W_i = np.random.randn(combined, hidden_size) * scale
        self.W_c = np.random.randn(combined, hidden_size) * scale
        self.W_o = np.random.randn(combined, hidden_size) * scale

        self.b_f = np.ones(hidden_size)   # Bias forget gate to 1 (remember by default)
        self.b_i = np.zeros(hidden_size)
        self.b_c = np.zeros(hidden_size)
        self.b_o = np.zeros(hidden_size)

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def forward(self, x_t, h_prev, c_prev):
        """One LSTM step. Returns (h_t, c_t) and gate values for visualisation."""
        combined = np.concatenate([h_prev, x_t])

        # Gates
        f_t = self.sigmoid(combined @ self.W_f + self.b_f)  # Forget gate
        i_t = self.sigmoid(combined @ self.W_i + self.b_i)  # Input gate
        c_tilde = np.tanh(combined @ self.W_c + self.b_c)   # Candidate
        o_t = self.sigmoid(combined @ self.W_o + self.b_o)  # Output gate

        # Cell state update (additive! - this is the key to solving vanishing gradients)
        c_t = f_t * c_prev + i_t * c_tilde

        # Hidden state
        h_t = o_t * np.tanh(c_t)

        return h_t, c_t, {'forget': f_t, 'input': i_t, 'output': o_t, 'candidate': c_tilde}

# Demo LSTM
input_size = 4
hidden_size = 3

lstm = LSTMCell(input_size, hidden_size)
X = np.random.randn(5, input_size)

h = np.zeros(hidden_size)
c = np.zeros(hidden_size)

print("LSTM Forward Pass:")
print(f"  Note: forget gate biased to 1.0 (remember by default)\n")

gate_history = {'forget': [], 'input': [], 'output': []}

for t in range(5):
    h, c, gates = lstm.forward(X[t], h, c)
    gate_history['forget'].append(gates['forget'])
    gate_history['input'].append(gates['input'])
    gate_history['output'].append(gates['output'])
    print(f"  t={t+1}: h={np.round(h, 3)}")
    print(f"         forget={np.round(gates['forget'], 3)}, input={np.round(gates['input'], 3)}, output={np.round(gates['output'], 3)}")

In [ ]:
# Visualise LSTM gate activations over time
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

gate_names = ['Forget Gate', 'Input Gate', 'Output Gate']
gate_keys = ['forget', 'input', 'output']
gate_colors = ['#e74c3c', '#2ecc71', '#3498db']

for ax, name, key, color in zip(axes, gate_names, gate_keys, gate_colors):
    data = np.array(gate_history[key])
    for dim in range(hidden_size):
        ax.plot(range(1, 6), data[:, dim], marker='o', label=f'dim {dim}',
                linewidth=2, alpha=0.8)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Gate Value (0-1)')
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('LSTM Gate Activations Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Gate values near 1.0 = 'open' (let information through)")
print("Gate values near 0.0 = 'closed' (block information)")
print("Forget gate near 1.0 = keep old cell state; near 0.0 = forget it")

---

# Part 4: Character-Level Language Model

## 4.1 The Task

A character-level language model predicts the **next character** given the previous characters. This is a classic RNN use case from Karpathy's famous blog post.

**Example:**
- Training text: `"hello"`
- Input sequence: `h → e → l → l`
- Target output:  `e → l → l → o`

The model learns to predict one character at a time, and at generation time we feed each predicted character back as the next input.

### How It Works

1. **Encode** each character as a one-hot vector
2. **Feed** characters one at a time to the RNN
3. **Output** a probability distribution over all characters
4. **Train** using cross-entropy loss
5. **Generate** by sampling from the output distribution

In [ ]:
# Character-level language model from scratch
class CharRNN:
    """Minimal character-level RNN language model."""

    def __init__(self, vocab_size, hidden_size, lr=0.01):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.lr = lr

        # Weight initialisation
        scale = 0.01
        self.W_xh = np.random.randn(vocab_size, hidden_size) * scale
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale
        self.W_hy = np.random.randn(hidden_size, vocab_size) * scale
        self.b_h = np.zeros(hidden_size)
        self.b_y = np.zeros(vocab_size)

    def forward(self, inputs, h_prev):
        """Forward pass through the sequence."""
        xs, hs, ys, ps = {}, {}, {}, {}
        hs[-1] = h_prev.copy()

        for t in range(len(inputs)):
            # One-hot encode input
            xs[t] = np.zeros(self.vocab_size)
            xs[t][inputs[t]] = 1

            # Hidden state
            hs[t] = np.tanh(xs[t] @ self.W_xh + hs[t-1] @ self.W_hh + self.b_h)

            # Output (logits)
            ys[t] = hs[t] @ self.W_hy + self.b_y

            # Softmax probabilities
            exp_ys = np.exp(ys[t] - np.max(ys[t]))
            ps[t] = exp_ys / np.sum(exp_ys)

        return xs, hs, ps

    def loss(self, ps, targets):
        """Cross-entropy loss."""
        return sum(-np.log(ps[t][targets[t]] + 1e-10) for t in range(len(targets)))

    def sample(self, h, seed_idx, n):
        """Generate n characters starting from seed_idx."""
        x = np.zeros(self.vocab_size)
        x[seed_idx] = 1
        result = []

        for _ in range(n):
            h = np.tanh(x @ self.W_xh + h @ self.W_hh + self.b_h)
            y = h @ self.W_hy + self.b_y
            exp_y = np.exp(y - np.max(y))
            p = exp_y / np.sum(exp_y)

            idx = np.random.choice(self.vocab_size, p=p)
            x = np.zeros(self.vocab_size)
            x[idx] = 1
            result.append(idx)

        return result

# Prepare training data
text = "hello world. hello deep learning. hello neural networks. "
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

print(f"Training text: '{text}'")
print(f"Vocabulary ({vocab_size} chars): {chars}")
print(f"\nCharacter encoding example:")
print(f"  'h' -> {char_to_idx['h']}")
print(f"  'e' -> {char_to_idx['e']}")
print(f"  'l' -> {char_to_idx['l']}")

In [ ]:
# Train the character-level model
hidden_size = 32
model = CharRNN(vocab_size, hidden_size, lr=0.1)

# Training parameters
seq_length = 15
data = [char_to_idx[ch] for ch in text]
losses = []

print("Training character-level RNN...")
print(f"  Sequence length: {seq_length}")
print(f"  Hidden size: {hidden_size}")
print()

for epoch in range(200):
    h = np.zeros(hidden_size)
    epoch_loss = 0
    n_batches = 0

    for p in range(0, len(data) - seq_length - 1, seq_length):
        inputs = data[p:p + seq_length]
        targets = data[p + 1:p + seq_length + 1]

        # Forward pass
        xs, hs, ps = model.forward(inputs, h)
        loss = model.loss(ps, targets)
        epoch_loss += loss
        n_batches += 1

        # Simple gradient descent (numerical gradients for clarity)
        # In practice you'd use BPTT (covered in Part 5)
        eps = 1e-5
        for param_name in ['W_xh', 'W_hh', 'W_hy', 'b_h', 'b_y']:
            param = getattr(model, param_name)
            grad = np.zeros_like(param)
            # Only compute gradients for a subset of params (for speed)
            if epoch < 200:
                it = np.nditer(param, flags=['multi_index'])
                count = 0
                while not it.finished and count < 20:
                    idx = it.multi_index
                    old_val = param[idx]

                    param[idx] = old_val + eps
                    _, _, ps_plus = model.forward(inputs, h)
                    loss_plus = model.loss(ps_plus, targets)

                    param[idx] = old_val - eps
                    _, _, ps_minus = model.forward(inputs, h)
                    loss_minus = model.loss(ps_minus, targets)

                    grad[idx] = (loss_plus - loss_minus) / (2 * eps)
                    param[idx] = old_val
                    it.iternext()
                    count += 1

            # Gradient clipping
            np.clip(grad, -5, 5, out=grad)
            param -= model.lr * grad

        h = hs[len(inputs) - 1]

    avg_loss = epoch_loss / max(n_batches, 1)
    losses.append(avg_loss)

    if epoch % 50 == 0:
        # Sample from the model
        sample_h = np.zeros(hidden_size)
        sample_ids = model.sample(sample_h, char_to_idx['h'], 40)
        sample_text = ''.join(idx_to_char[i] for i in sample_ids)
        print(f"  Epoch {epoch:3d} | Loss: {avg_loss:.2f} | Sample: 'h{sample_text}'")

print("\nTraining complete!")

In [ ]:
# Plot training loss and generate final samples
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax = axes[0]
ax.plot(losses, linewidth=2, color='#e74c3c')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Average Loss', fontsize=12)
ax.set_title('Character-Level RNN Training Loss', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Generate multiple samples
ax = axes[1]
ax.axis('off')
ax.set_title('Generated Text Samples', fontsize=13, fontweight='bold')

samples_text = []
for i, seed_char in enumerate(['h', 'l', 'd', 'n']):
    sample_h = np.zeros(hidden_size)
    sample_ids = model.sample(sample_h, char_to_idx.get(seed_char, 0), 30)
    generated = seed_char + ''.join(idx_to_char[idx] for idx in sample_ids)
    samples_text.append(f"Seed '{seed_char}': {generated}")

for i, s in enumerate(samples_text):
    ax.text(0.05, 0.85 - i * 0.22, s, fontsize=11, fontfamily='monospace',
            transform=ax.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("The model learns character-level patterns from the training text.")
print("With more data and training, it would produce more coherent text.")

---

# Part 5: Backpropagation in RNN (BPTT)

## 5.1 Backpropagation Through Time

Training an RNN uses **Backpropagation Through Time (BPTT)** - standard backpropagation applied to the unrolled network.

### The Key Challenge

When we unroll the RNN, the **same weights** appear at every time step. The total gradient for a weight is the **sum of gradients from all time steps**.

```
Total gradient: ∂L/∂W = Σ_t (∂L_t/∂W)
```

### BPTT Algorithm

1. **Forward pass:** Compute all hidden states h_1, h_2, ..., h_T and outputs
2. **Compute loss:** Sum losses at each time step
3. **Backward pass:** Starting from time T, propagate gradients back through time
4. **Accumulate gradients:** Sum gradients for shared weights across all time steps
5. **Update weights:** Apply gradient descent

### The Vanishing/Exploding Gradient Problem

During BPTT, the gradient at time step `t` depends on all future time steps:

```
∂L/∂h_t = Σ_{k=t}^{T} (∂L_k/∂h_t)

∂h_k/∂h_t = Π_{i=t+1}^{k} (∂h_i/∂h_{i-1}) = Π_{i=t+1}^{k} diag(1 - h_i²) · W_hh
```

This product of matrices either **shrinks** (vanishing) or **grows** (exploding) exponentially with the number of time steps `k - t`.

### Solutions

| Problem | Solution |
|---------|----------|
| **Vanishing gradients** | LSTM/GRU (additive cell state updates) |
| **Exploding gradients** | Gradient clipping: `if ||g|| > threshold: g = threshold * g / ||g||` |
| **Computational cost** | Truncated BPTT: only backprop through last K steps |

In [ ]:
# Demonstrate vanishing gradients in vanilla RNN vs LSTM-style updates
def simulate_gradient_flow(seq_length, mode='vanilla'):
    """Simulate how gradients flow backward through time."""
    np.random.seed(42)
    hidden_size = 10

    if mode == 'vanilla':
        # Vanilla RNN: gradient is product of W_hh and diag(1 - tanh²)
        W_hh = np.random.randn(hidden_size, hidden_size) * 0.5
        gradient_norms = [1.0]

        grad = np.eye(hidden_size)
        for t in range(seq_length):
            # Simulate tanh derivative (values between 0 and 1)
            tanh_deriv = np.diag(np.random.uniform(0.1, 0.9, hidden_size))
            grad = tanh_deriv @ W_hh @ grad
            gradient_norms.append(np.linalg.norm(grad))

    elif mode == 'lstm':
        # LSTM: gradient flows through cell state with additive updates
        gradient_norms = [1.0]
        grad = 1.0

        for t in range(seq_length):
            # Forget gate (close to 1 = gradients flow easily)
            f_t = np.random.uniform(0.85, 0.99)
            grad = grad * f_t  # Additive path preserves gradients
            gradient_norms.append(abs(grad))

    return gradient_norms

# Compare gradient flow
seq_lengths = range(1, 51)

vanilla_final = []
lstm_final = []

for sl in seq_lengths:
    v_norms = simulate_gradient_flow(sl, 'vanilla')
    l_norms = simulate_gradient_flow(sl, 'lstm')
    vanilla_final.append(v_norms[-1])
    lstm_final.append(l_norms[-1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: gradient magnitude vs sequence length
ax = axes[0]
ax.semilogy(seq_lengths, vanilla_final, 'r-', linewidth=2, label='Vanilla RNN')
ax.semilogy(seq_lengths, lstm_final, 'b-', linewidth=2, label='LSTM')
ax.set_xlabel('Sequence Length (time steps back)', fontsize=12)
ax.set_ylabel('Gradient Magnitude (log scale)', fontsize=12)
ax.set_title('Vanishing Gradient: Vanilla RNN vs LSTM', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Right: detailed gradient flow for one sequence
ax = axes[1]
v_norms = simulate_gradient_flow(30, 'vanilla')
l_norms = simulate_gradient_flow(30, 'lstm')
ax.plot(range(31), v_norms, 'r-o', markersize=3, linewidth=2, label='Vanilla RNN', alpha=0.8)
ax.plot(range(31), l_norms, 'b-o', markersize=3, linewidth=2, label='LSTM', alpha=0.8)
ax.set_xlabel('Steps Back in Time', fontsize=12)
ax.set_ylabel('Gradient Magnitude', fontsize=12)
ax.set_title('Gradient Flow (30-step sequence)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Vanilla RNN: gradients vanish exponentially - can't learn long-range dependencies")
print("LSTM: gradients decay slowly through the additive cell state path")

## 5.2 Gradient Clipping

A practical technique to handle **exploding gradients**:

```python
# If gradient norm exceeds threshold, scale it down
if ||gradient|| > max_norm:
    gradient = max_norm * gradient / ||gradient||
```

This preserves the gradient **direction** but limits its **magnitude**.

In [ ]:
# Demonstrate gradient clipping
def clip_gradient(grad, max_norm=5.0):
    """Clip gradient by global norm."""
    norm = np.linalg.norm(grad)
    if norm > max_norm:
        grad = max_norm * grad / norm
    return grad, norm

# Simulate gradients during training
np.random.seed(42)
raw_norms = []
clipped_norms = []

for step in range(50):
    # Simulate a gradient (occasionally exploding)
    grad = np.random.randn(10) * (1 + 3 * np.random.random())
    if step % 7 == 0:  # Simulate gradient explosion
        grad *= 10

    raw_norm = np.linalg.norm(grad)
    clipped_grad, _ = clip_gradient(grad, max_norm=5.0)
    clipped_norm = np.linalg.norm(clipped_grad)

    raw_norms.append(raw_norm)
    clipped_norms.append(clipped_norm)

plt.figure(figsize=(12, 4))
plt.plot(raw_norms, 'r-', alpha=0.7, linewidth=2, label='Raw gradients')
plt.plot(clipped_norms, 'b-', linewidth=2, label='Clipped gradients (max=5)')
plt.axhline(y=5.0, color='gray', linestyle='--', alpha=0.5, label='Clip threshold')
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Gradient Norm', fontsize=12)
plt.title('Gradient Clipping in Action', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Gradient clipping prevents exploding gradients while preserving direction")
print("Common max_norm values: 1.0, 5.0, 10.0")

---

# Part 6: Stacking Several Layers - Deep RNNs

## 6.1 Why Go Deep?

A single RNN layer learns one level of representation. **Stacking multiple layers** allows the network to learn hierarchical features:

- **Layer 1:** Low-level patterns (e.g., character n-grams, basic phonemes)
- **Layer 2:** Mid-level patterns (e.g., word-level features, phrases)
- **Layer 3:** High-level patterns (e.g., sentence structure, semantics)

### Architecture

```
Output:     y_1      y_2      y_3
             ↑        ↑        ↑
Layer 3:  [RNN] → [RNN] → [RNN]     ← High-level features
             ↑        ↑        ↑
Layer 2:  [RNN] → [RNN] → [RNN]     ← Mid-level features
             ↑        ↑        ↑
Layer 1:  [RNN] → [RNN] → [RNN]     ← Low-level features
             ↑        ↑        ↑
Input:      x_1      x_2      x_3
```

### Key Implementation Detail

All layers except the last must use `return_sequences=True` so they output a sequence (not just the final hidden state) for the next layer to consume.

### Practical Guidelines

- **2-3 layers** is usually sufficient
- **Dropout between layers** prevents overfitting
- **Decreasing hidden size** is common (e.g., 256 → 128 → 64)
- More layers = more parameters = more data needed

In [ ]:
# Implement a deep (stacked) RNN from scratch
class DeepRNN:
    """Multi-layer RNN to demonstrate stacking."""

    def __init__(self, input_size, hidden_sizes):
        """
        Args:
            input_size: dimension of input features
            hidden_sizes: list of hidden sizes for each layer, e.g. [128, 64, 32]
        """
        self.layers = []
        self.hidden_sizes = hidden_sizes

        prev_size = input_size
        for hs in hidden_sizes:
            self.layers.append(VanillaRNNCell(prev_size, hs))
            prev_size = hs  # Output of this layer is input to next

    def forward_sequence(self, X):
        """Process sequence through all layers. Returns hidden states per layer."""
        T = X.shape[0]
        all_layer_states = []

        current_input = X
        for layer_idx, (layer, hs) in enumerate(zip(self.layers, self.hidden_sizes)):
            h = np.zeros(hs)
            layer_states = []

            for t in range(T):
                h = layer.forward(current_input[t], h)
                layer_states.append(h.copy())

            layer_states = np.array(layer_states)
            all_layer_states.append(layer_states)
            current_input = layer_states  # Feed to next layer

        return all_layer_states

# Create a 3-layer deep RNN
deep_rnn = DeepRNN(input_size=4, hidden_sizes=[16, 8, 4])

# Process a sequence
X = np.random.randn(10, 4)  # 10 time steps, 4 features
layer_outputs = deep_rnn.forward_sequence(X)

print("Deep RNN Architecture:")
print(f"  Input: ({X.shape[0]} steps, {X.shape[1]} features)")
for i, (states, hs) in enumerate(zip(layer_outputs, [16, 8, 4])):
    print(f"  Layer {i+1}: hidden_size={hs}, output shape={states.shape}")
print()

# Visualise what each layer captures
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, (ax, states) in enumerate(zip(axes, layer_outputs)):
    im = ax.imshow(states.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xlabel('Time Step', fontsize=11)
    ax.set_ylabel('Hidden Dimension', fontsize=11)
    ax.set_title(f'Layer {i+1} (hidden={states.shape[1]})', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Deep RNN: Hidden States at Each Layer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each layer learns increasingly abstract representations of the input sequence")
print("Layer 1 responds to raw input patterns; Layer 3 captures higher-level structure")

---

# Part 7: Visualising RNNs

## 7.1 What Do RNNs Learn?

Understanding what an RNN has learned is crucial for debugging and building intuition. Key visualisation techniques:

1. **Hidden state trajectories** - how the internal state evolves
2. **Gate activations** - what the LSTM/GRU gates are doing
3. **Hidden unit activations** - individual neurons that detect specific patterns
4. **t-SNE / PCA of hidden states** - clustering in hidden space

### Karpathy's Insights (2015)

In "Visualizing and Understanding Recurrent Networks", Karpathy showed that individual LSTM cells learn interpretable features:
- Some cells track **quote depth** (open/close quotes)
- Some cells track **line length** (position in a line)
- Some cells activate on **specific characters** or patterns
- Some cells detect **if/else blocks** in code

In [ ]:
# Visualisation 1: Hidden state trajectories with PCA
from numpy.linalg import svd

# Process a meaningful sequence through our RNN
text_seq = "hello world hello deep learning"
# Encode as simple integer sequence
unique_chars = sorted(set(text_seq))
ch2id = {c: i for i, c in enumerate(unique_chars)}
encoded = np.array([ch2id[c] for c in text_seq])

# One-hot encode
n_chars = len(unique_chars)
X_onehot = np.zeros((len(encoded), n_chars))
for i, idx in enumerate(encoded):
    X_onehot[i, idx] = 1

# Process through RNN
rnn_vis = VanillaRNNCell(n_chars, 16)
states = rnn_vis.forward_sequence(X_onehot)
states = states[1:]  # Remove initial zero state

# PCA to 2D for visualisation
states_centered = states - states.mean(axis=0)
U, S, Vt = svd(states_centered, full_matrices=False)
states_2d = states_centered @ Vt[:2].T

# Plot hidden state trajectory
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: 2D trajectory
ax = axes[0]
# Color by position in sequence
colors = plt.cm.viridis(np.linspace(0, 1, len(text_seq)))
for i in range(len(text_seq) - 1):
    ax.annotate('', xy=(states_2d[i+1, 0], states_2d[i+1, 1]),
                xytext=(states_2d[i, 0], states_2d[i, 1]),
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=1.5))

for i, ch in enumerate(text_seq):
    ax.scatter(states_2d[i, 0], states_2d[i, 1], c=[colors[i]], s=80, zorder=5, edgecolors='black', linewidth=0.5)
    ax.annotate(ch, (states_2d[i, 0], states_2d[i, 1]),
                textcoords="offset points", xytext=(5, 5), fontsize=9)

ax.set_xlabel('PC1', fontsize=12)
ax.set_ylabel('PC2', fontsize=12)
ax.set_title('Hidden State Trajectory (PCA)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Right: hidden state heatmap
ax = axes[1]
im = ax.imshow(states[:, :8].T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('Character Position', fontsize=12)
ax.set_ylabel('Hidden Unit', fontsize=12)
ax.set_title('Hidden Unit Activations', fontsize=13, fontweight='bold')

# Add character labels
ax.set_xticks(range(0, len(text_seq), 2))
ax.set_xticklabels([text_seq[i] for i in range(0, len(text_seq), 2)], fontsize=8)
plt.colorbar(im, ax=ax, label='Activation')

plt.tight_layout()
plt.show()

print("Left: Same characters cluster together in hidden space (e.g., all 'l's are nearby)")
print("Right: Each hidden unit responds differently to different characters")

In [ ]:
# Visualisation 2: Character-sensitive hidden units
# Find which hidden units respond most to specific characters

char_responses = {}
for ch in unique_chars:
    # Find all positions where this character appears
    positions = [i for i, c in enumerate(text_seq) if c == ch]
    if positions:
        # Average hidden state when this character is the input
        avg_state = np.mean(states[positions], axis=0)
        char_responses[ch] = avg_state

# Plot character selectivity
fig, ax = plt.subplots(figsize=(12, 5))

chars_to_show = [c for c in unique_chars if c != ' ']
response_matrix = np.array([char_responses[c][:8] for c in chars_to_show])

im = ax.imshow(response_matrix, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_yticks(range(len(chars_to_show)))
ax.set_yticklabels([f"'{c}'" for c in chars_to_show], fontsize=11)
ax.set_xticks(range(8))
ax.set_xticklabels([f'Unit {i}' for i in range(8)], fontsize=10)
ax.set_xlabel('Hidden Unit', fontsize=12)
ax.set_ylabel('Character', fontsize=12)
ax.set_title('Character Selectivity of Hidden Units', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Average Activation')

plt.tight_layout()
plt.show()

print("Each hidden unit develops a 'preference' for certain characters")
print("This is how the RNN builds internal representations of the input")

In [ ]:
# Visualisation 3: How hidden state changes with context
# Same character 'l' appears in different contexts

l_positions = [i for i, c in enumerate(text_seq) if c == 'l']
print("Character 'l' appears at positions:", l_positions)
print("Contexts:")
for pos in l_positions:
    start = max(0, pos - 3)
    end = min(len(text_seq), pos + 4)
    context = text_seq[start:end]
    marker = ' ' * (pos - start) + '^'
    print(f"  ...{context}...")
    print(f"     {marker}")

# Compare hidden states at each 'l'
fig, ax = plt.subplots(figsize=(10, 4))

l_states = states[l_positions, :8]
x_labels = [f"pos {p}: ...{text_seq[max(0,p-2):p+3]}..." for p in l_positions]

im = ax.imshow(l_states, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_yticks(range(len(l_positions)))
ax.set_yticklabels(x_labels, fontsize=10)
ax.set_xticks(range(8))
ax.set_xticklabels([f'h[{i}]' for i in range(8)], fontsize=10)
ax.set_title("Hidden State for 'l' in Different Contexts", fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Activation')

plt.tight_layout()
plt.show()

print("\nKey insight: The same character produces DIFFERENT hidden states")
print("depending on context. This is how RNNs capture sequential dependencies.")

---

## Exercises

Test your understanding with these hands-on exercises. Try to solve them before looking at the hints.

### Exercise 1: RNN Parameter Counting

For a vanilla RNN with `input_size=10` and `hidden_size=20`:
1. How many parameters does `W_xh` have?
2. How many parameters does `W_hh` have?
3. How many total parameters (including biases)?
4. How does this compare to an LSTM with the same dimensions?

Work it out by hand, then verify with code.

In [ ]:
# Exercise 1: Parameter counting
# Vanilla RNN: W_xh (input_size x hidden_size) + W_hh (hidden_size x hidden_size) + b_h (hidden_size)
# LSTM: 4x the vanilla RNN parameters (one set for each gate: forget, input, candidate, output)

input_size = 10
hidden_size = 20

# Your calculations here:
# W_xh_params = ?
# W_hh_params = ?
# b_h_params = ?
# vanilla_total = ?
# lstm_total = ?

# Verify:
# print(f"Vanilla RNN: {vanilla_total} parameters")
# print(f"LSTM: {lstm_total} parameters")
# print(f"LSTM has {lstm_total / vanilla_total}x more parameters")

### Exercise 2: Implement Truncated BPTT

Modify the character-level language model to use truncated BPTT with a configurable truncation length K. Compare training with K=5, K=15, and K=30. How does truncation length affect:
1. Training speed?
2. Final loss?
3. Quality of generated text?

In [ ]:
# Exercise 2: Truncated BPTT
# Hint: Instead of backpropagating through the entire sequence,
# only backpropagate through the last K time steps.
# This trades off long-range learning for computational efficiency.

# K = 5   # Short truncation - fast but misses long patterns
# K = 15  # Medium truncation - good balance
# K = 30  # Long truncation - slower but captures more context

# Your code here


### Exercise 3: Visualise Gate Dynamics

Using the LSTM cell implementation from Part 3, process the text "aaaaabbbbb" (5 a's followed by 5 b's). Visualise the forget gate and input gate activations. What do you expect to see when the character changes from 'a' to 'b'?

In [ ]:
# Exercise 3: Gate dynamics at character transitions
# Hint: The forget gate should activate (close to 0) when the pattern changes
# The input gate should activate (close to 1) to store the new pattern

# text = "aaaaabbbbb"
# Process through LSTM and plot gate activations
# Look for changes at position 5 (the transition point)

# Your code here


---

## Key Takeaways

**Relevant UoA Course:** COMPSCI 714 (Lecture 4)

1. **Sequential data** has temporal/positional dependencies - order matters
2. RNN tasks: one-to-one, one-to-many, many-to-one, many-to-many
3. RNN equation: `h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b_h)` - same weights at every step
4. **Hidden state** `h_t` encodes a summary of the sequence up to time `t`
5. **LSTM** solves vanishing gradients with gates and additive cell state updates
6. **GRU** is a simpler alternative with 2 gates instead of 3
7. **Character-level language models** predict the next character given context
8. **BPTT** = backpropagation applied to the unrolled RNN; gradients vanish/explode
9. **Deep RNNs** stack layers for hierarchical feature learning
10. **Visualising** hidden states reveals what the RNN has learned

---

## Exam Preparation Guide

### Essential Concepts

- Trace an RNN forward pass: given x_1, x_2, x_3 and weights, compute h_1, h_2, h_3
- Explain why vanilla RNNs fail on long sequences (vanishing gradient via repeated W_hh multiplication)
- Describe each LSTM gate and its role
- Calculate parameter counts: Vanilla RNN vs LSTM vs GRU
- Explain BPTT and why gradients vanish/explode
- Know when to use deep (stacked) RNNs

### Common Exam Questions

1. "Draw the unrolled RNN for a sequence of length 4. Label all weights and states."
2. "Explain how the LSTM forget gate solves the vanishing gradient problem."
3. "Compare the number of parameters in a vanilla RNN, LSTM, and GRU with hidden_size=64."
4. "What is truncated BPTT and why is it used?"
5. "Given this sequence task, which RNN architecture would you choose and why?"

### Common Mistakes to Avoid

- Forgetting that RNN weights are **shared** across time steps
- Confusing LSTM **hidden state** (h_t) with **cell state** (C_t)
- Not understanding that LSTM has 4x parameters of vanilla RNN
- Thinking bidirectional RNNs can be used for real-time prediction (they can't - need full sequence)

---

---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (B10a - COMPSCI 714 Lecture 4)
- [ ] Sequential data and why order matters
- [ ] RNN task types (one-to-one, one-to-many, many-to-one, many-to-many)
- [ ] RNN architecture: hidden state, weight sharing, unrolling
- [ ] LSTM gates: forget, input, output, cell state
- [ ] GRU: reset and update gates
- [ ] Character-level language model
- [ ] BPTT and vanishing/exploding gradients
- [ ] Deep (stacked) RNNs
- [ ] Visualising RNN hidden states

---